In [1]:
import math
import numpy as np

def prom_update(prom_prev: float, x_new: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return x_new
    else:
        return prom_prev + (x_new - prom_prev) / n_curr


def s_cuad_update(s_cuad_prev: float, prom_prev: float, prom_curr: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return 0.0
    else:
        return (1 - (1/(n_curr - 1))) * s_cuad_prev + n_curr * ((prom_curr - prom_prev)**2)

### Ejercicio 9
Considerar un centro de diagnóstico por imágenes que cuenta con dos etapas de atención en serie:
- En la primera etapa, los pacientes realizan el proceso de admisión administrativa.
- Luego pasan a una única sala de estudios donde se realiza el diagnóstico.

Los pacientes llegan al sistema de acuerdo con un proceso de Poisson no homogéneo cuya tasa varía a lo largo del día. Cada jornada tiene $16$ horas de atención a pacientes. La función de intensidad es inicialmente de $4$ pacientes por hora, aumenta linealmente hasta alcanzar $14$ pacientes por hora luego de $4$ horas, y posteriormente disminuye linealmente hasta volver a $4$ pacientes por hora luego de otras $4$ horas. Este comportamiento se repite periódicamente; es decir,
$$ \lambda(t + 8) = \lambda(t), \quad 0 \le t \le 8 $$
donde t se mide en horas.

Suponer que:
- El centro recibe pacientes únicamente durante las primeras $16$ horas de operación. Luego de ese instante no ingresan nuevos pacientes, pero aquellos que ya ingresaron continúan siendo atendidos hasta completar el proceso.
- Los tiempos de atención en admisión se distribuyen exponencialmente con tasa de $15$ pacientes por hora.
- Los tiempos de realización de estudios se distribuyen exponencialmente con tasa $12$ pacientes por hora.
- El siguiente paciente en ser atendido es el que más tiempo estuvo en espera.

Se pide:

a. Desarrollar un programa que simule el proceso durante una jornada de 16 horas, registrando los tiempos de llegada, los tiempos de servicio y la evolución del número de trabajos en cola.

b. Estimar el tiempo promedio de permanencia en el sistema de los pacientes atendidos durante la jornada. Realizar simulaciones independientes hasta que la desviación estándar muestral del estimador de la media sea menor que $0.01$.

c. Estimar la probabilidad de que, luego de las $16$ horas, todavía queden pacientes en el sistema. Detener las simulaciones cuando la desviación estándar muestral del estimador de la proporción sea menor que $0.005$.

d. Estimar el tiempo esperado adicional necesario para finalizar la atención de todos los pacientes que ingresaron antes del cierre. Detener las simulaciones cuando la desviación estándar muestral del estimador de la media sea menor que $0.01$. Expresar el resultado en minutos.

e. Construir intervalos de confianza del $95\%$ para las cantidades estimadas en los incisos anteriores.

---

In [2]:
# Punto A

def lamda_t(t: float):
    t_m = t % 8

    if t_m <= 4:
        return 4 + 2.5 * t_m
    else:
        return 24 - 2.5 * t_m

def t_llegada_paciente():
    lamda_cota = 14
    t = 0

    while True:
        t -= math.log(1 - np.random.random()) / lamda_cota
        V = np.random.random()

        if V < lamda_t(t) / lamda_cota:
            return t

def t_atencion():
    # Exponencial con lambda = 15 (15 pacientes por hora)
    return -math.log(1 - np.random.random()) / 15

def t_estudio():
    # Exponencial con lambda = 12 (12 pacientes por hora)
    return -math.log(1 - np.random.random()) / 12


def punto_a(T: int = 16):
    # Registros
    A1, A2 = [], [] # Tiempo de atención y del estudio respectivamente
    D = []  # Tiempo de salida de un paciente
    N = [(0, 0)]  # Cola de pacientes
    N_T = [0]  # Tiempo en el cual se cambia la cola de pacientes

    # Inicialización
    t = 0
    n1, n2 = 0, 0

    tA = t + t_llegada_paciente()
    t1 = math.inf
    t2 = math.inf

    while tA < math.inf or t1 < math.inf or t2 < math.inf:
        evento_prox = min(tA, t1, t2)

        if evento_prox == tA:
            t = tA
            n1 += 1

            tA = t + t_llegada_paciente()
            if tA > T:
                # La jornada actual supero T, por lo que no se reciben nuevos
                # pacientes y se termina el trabajo con los pacientes en la cola
                tA = math.inf

            if n1 == 1:
                t1 = t + t_atencion()

            A1.append(t)
            N.append((n1, n2))
            N_T.append(t)

        elif evento_prox == t1:
            t = t1
            n1 -= 1

            if n1 == 0:
                t1 = math.inf
            else:
                t1 = t + t_atencion()

            n2 += 1
            if n2 == 1:
                t2 = t + t_estudio()

            A2.append(t)
            N.append((n1, n2))
            N_T.append(t)

        else:
            # evento_prox == t2

            t = t2
            n2 -= 1

            if n2 == 0:
                t2 = math.inf
            else:
                t2 = t + t_estudio()

            D.append(t)
            N.append((n1, n2))
            N_T.append(t)

    return t, A1, A2, D, N, N_T

In [3]:
# Punto B

def generar_media_i():
    _, A1_i, _, D_i, _, _ = punto_a()

    return (np.array(D_i) - np.array(A1_i)).mean()


def punto_b():
    media = generar_media_i()
    s_cuad, n = 0, 1

    while n < 100 or math.sqrt(s_cuad / n) >= 0.01:
        X = generar_media_i()
        n += 1

        media_prev = media
        media = prom_update(media, X, n)
        s_cuad = s_cuad_update(s_cuad, media_prev, media, n)

    print(f"Cantidad de datos generados: ", n)
    print(f"Los pacientes permanecen en el sistema aprox {media:.4f} horas")
    print(f"Desviación estándar muestral = {math.sqrt(s_cuad / n):.4f} ")


punto_b()

Cantidad de datos generados:  100
Los pacientes permanecen en el sistema aprox 0.2179 horas
Desviación estándar muestral = 0.0041 


In [4]:
# Punto C

def generar_p_i():
    _, _, _, D_i, _, _ = punto_a()

    return (np.array(D_i) >= 16.0).mean()


def punto_c():
    p = generar_p_i()
    n = 1

    while n < 100 or math.sqrt(p * (1 - p) / n) >= 0.005:
        X = generar_p_i()
        n += 1

        p = prom_update(p, X, n)

    print(f"Cantidad de datos generados: ", n)
    print(f"La probabilidad de que todavía queden pacientes en espera al finalizar la jornada es aprox. {p:.4f}")
    print(f"Desviación estándar muestral = {math.sqrt(p * (1 - p) / n):.4f} ")


punto_c()

Cantidad de datos generados:  534
La probabilidad de que todavía queden pacientes en espera al finalizar la jornada es aprox. 0.0135
Desviación estándar muestral = 0.0050 


In [5]:
# Punto D

def generar_media_i():
    _, A1_i, _, D_i, _, _ = punto_a()

    df_D_i = np.array(D_i)
    df_A1_i = np.array(A1_i)

    p_gt_16 = np.count_nonzero(df_D_i > 16.0)

    if p_gt_16 == 0:
        return 0

    t_atencion_adicional = df_D_i[-p_gt_16:] - df_A1_i[-p_gt_16:]

    return sum(t_atencion_adicional)


def punto_d():
    media = generar_media_i()
    s_cuad, n = 0, 1

    while n < 100 or math.sqrt(s_cuad / n) >= 0.005:
        X = generar_media_i()
        n += 1

        media_prev = media
        media = prom_update(media, X, n)
        s_cuad = s_cuad_update(s_cuad, media_prev, media, n)

    print(f"Cantidad de datos generados: ", n)
    print(f"Tiempo promedio de atención adicional, por terminar la jornada es aprox {media * 60:.4f} minutos")
    print(f"Desviación estándar muestral = {math.sqrt(s_cuad / n):.4f} ")


punto_d()

Cantidad de datos generados:  14803
Tiempo promedio de atención adicional, por terminar la jornada es aprox 19.9758 minutos
Desviación estándar muestral = 0.0050 
